# Training a microWakeWord Model

This notebook steps you through training a basic microWakeWord model. It is intended as a **starting point** for advanced users. You should use Python 3.10.

**The model generated will most likely not be usable for everyday use; it may be difficult to trigger or falsely activates too frequently. You will most likely have to experiment with many different settings to obtain a decent model!**

In the comment at the start of certain blocks, I note some specific settings to consider modifying.

This runs on Google Colab, but is extremely slow compared to training on a local GPU. If you must use Colab, be sure to Change the runtime type to a GPU. Even then, it still slow!

At the end of this notebook, you will be able to download a tflite file. To use this in ESPHome, you need to write a model manifest JSON file. See the [ESPHome documentation](https://esphome.io/components/micro_wake_word) for the details and the [model repo](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2) for examples.

In [ ]:
# Installs microWakeWord. Be sure to restart the session after this is finished.
import platform

if platform.system() == "Darwin":
    # `pymicro-features` is installed from a fork to support building on macOS
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# `audio-metadata` is installed from a fork to unpin `attrs` from a version that breaks Jupyter
!pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

!git clone https://github.com/kahrendt/microWakeWord
!pip install -e ./microWakeWord
# Pinned: datasets>=4.0 switched audio decoding to a torchcodec-based
# object instead of the old {'path':..., 'array':...} dict, which breaks
# several cells below that still use the old dict-style access. Installed
# AFTER the line above on purpose, so it overrides whatever version
# microWakeWord's own install just pulled in.
!pip install -q "datasets==3.6.0"


In [2]:
target_word = 'Hey REX'  # Phonetic spellings may produce better samples

import os
import sys

from IPython.display import Audio

# Force the known-working fork — the mainline rhasspy/piper-sample-generator
# repo has since been restructured (generate_samples.py no longer exists
# there).
!rm -rf piper-sample-generator
!git clone -b mps-support https://github.com/kahrendt/piper-sample-generator

!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

# piper-phonemize-cross was never published for Python 3.13 (Colab's
# current runtime) at all — no version pin fixes that. This is a
# maintained replacement that provides the identical importable module
# (piper_phonemize), so nothing else in the script needs to change.
!pip install -q torch torchaudio deep-phonemizer "piper_phonemize==1.3.0" -f https://k2-fsa.github.io/icefall/piper_phonemize.html

# The script's model loading (torch.load(model_path), one call, no other
# args) predates PyTorch tightening its default loading behavior in 2.6+
# — your Colab session has 2.11 installed, well past that point — so patch
# it to explicitly allow the older-style checkpoint this model ships as.
!sed -i 's/torch\.load(model_path)/torch.load(model_path, weights_only=False)/' piper-sample-generator/generate_samples.py

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")

!python3 piper-sample-generator/generate_samples.py "{target_word}" \
--max-samples 1 \
--batch-size 1 \
--output-dir generated_samples

Audio("generated_samples/0.wav", autoplay=True)



Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 135 (delta 53), reused 44 (delta 44), pack-reused 65 (from 1)
Receiving objects: 100% (135/135), 1.03 MiB | 3.05 MiB/s, done.
Resolving deltas: 100% (61/61), done.
--2026-09-29 13:37:09--  https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/642029941/73f4af3c-7cf8-4547-a7b9-3bd29e7f3c33?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-09-29T14%3A21%3A12Z&rscd=attachment%3B+filename%3Den_US-libritts_r-medium.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-4

In [3]:
# Generates a larger amount of wake word samples.
# Start here when trying to improve your model.
# See https://github.com/rhasspy/piper-sample-generator for the full set of
# parameters. In particular, experiment with noise-scales and noise-scale-ws,
# generating negative samples similar to the wake word, and generating many more
# wake word samples, possibly with different phonetic pronunciations.

!python3 piper-sample-generator/generate_samples.py "{target_word}" \
--max-samples 1000 \
--batch-size 100 \
--output-dir generated_samples

DEBUG:__main__:Loading /content/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/10 complete
DEBUG:__main__:Batch 2/10 complete
DEBUG:__main__:Batch 3/10 complete
DEBUG:__main__:Batch 4/10 complete
DEBUG:__main__:Batch 5/10 complete
DEBUG:__main__:Batch 6/10 complete
DEBUG:__main__:Batch 7/10 complete
DEBUG:__main__:Batch 8/10 complete
DEBUG:__main__:Batch 9/10 complete
DEBUG:__main__:Batch 10/10 complete
INFO:__main__:Done


In [4]:
!pip install -q "datasets==3.6.0"


In [5]:
!rm -rf mit_rirs


In [6]:
# Downloads audio data for augmentation. This can be slow!
# Borrowed from openWakeWord's automatic_model_training.ipynb, accessed March 4, 2024
#
# **Important note!** The data downloaded here has a mixture of difference
# licenses and usage restrictions. As such, any custom models trained with this
# data should be considered as appropriate for **non-commercial** personal use only.


import datasets
import scipy
import os

import numpy as np

from pathlib import Path
from tqdm import tqdm

## Download MIR RIR data

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    # Save clips to 16-bit PCM wav files
    for row in tqdm(rir_dataset):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

    fname = "bal_train09.tar"
    out_dir = f"audioset/{fname}"
    link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
    !wget -O {out_dir} {link}
    !cd audioset && tar -xf bal_train09.tar

    output_dir = "./audioset_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files
    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# Free Music Archive dataset
# https://github.com/mdeff/fma
# (Third-party mchl914 extra small set)

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fname = "fma_xs.zip"
    link = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/" + fname
    out_dir = f"fma/{fname}"
    !wget -O {out_dir} {link}
    !cd {output_dir} && unzip -q {fname}

    output_dir = "./fma_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Save clips to 16-bit PCM wav files
    fma_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("fma/fma_small").glob("**/*.mp3")]})
    fma_dataset = fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(fma_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))


Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

270it [01:15,  3.56it/s]


In [7]:
!ls mit_rirs | wc -l
!ls mit_rirs | head -5


270
h001_Bedroom_65txts.wav
h002_Bedroom_62txts.wav
h003_Office_LargeBrickWalledOpenPlanOffice_56txts.wav
h004_LivingRoom_Large_48txts.wav
h005_Office_Small_44txts.wav


In [8]:
# Sets up the augmentations.
# To improve your model, experiment with these settings and use more sources of
# background clips.

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(input_directory='generated_samples',
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )
augmenter = Augmentation(augmentation_duration_s=3.2,
                         augmentation_probabilities = {
                                "SevenBandParametricEQ": 0.1,
                                "TanhDistortion": 0.1,
                                "PitchShift": 0.1,
                                "BandStopFilter": 0.1,
                                "AddColorNoise": 0.1,
                                "AddBackgroundNoise": 0.75,
                                "Gain": 1.0,
                                "RIR": 0.5,
                            },
                         impulse_paths = ['mit_rirs'],
                         background_paths = ['fma_16k', 'audioset_16k'],
                         background_min_snr_db = -5,
                         background_max_snr_db = 10,
                         min_jitter_s = 0.195,
                         max_jitter_s = 0.205,
                         )


In [9]:
!rm -rf generated_augmented_features


In [10]:
# Augment a random clip and play it back to verify it works well

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, 'augmented_clip.wav')

Audio("augmented_clip.wav", autoplay=True)

In [11]:
# Augment samples and save the training, validation, and testing sets.
# Validating and testing samples generated the same way can make the model
# benchmark better than it performs in real-word use. Use real samples or TTS
# samples generated with a different TTS engine to potentially get more accurate
# benchmarks.

import os
from mmap_ninja.ragged import RaggedMmap

output_dir = 'generated_augmented_features'

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
  out_dir = os.path.join(output_dir, split)
  if not os.path.exists(out_dir):
      os.mkdir(out_dir)


  split_name = "train"
  repetition = 2

  spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=10,    # Uses the same spectrogram repeatedly, just shifted over by one frame. This simulates the streaming inferences while training/validating in nonstreaming mode.
                                     step_ms=10,
                                     )
  if split == "validation":
    split_name = "validation"
    repetition = 1
  elif split == "testing":
    split_name = "test"
    repetition = 1
    spectrograms = SpectrogramGeneration(clips=clips,
                                     augmenter=augmenter,
                                     slide_frames=1,    # The testing set uses the streaming version of the model, so no artificial repetition is necessary
                                     step_ms=10,
                                     )

  RaggedMmap.from_generator(
      out_dir=os.path.join(out_dir, 'wakeword_mmap'),
      sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
      batch_size=100,
      verbose=True,
  )

0it [00:00, ?it/s]

0it [00:00, ?it/s]

0it [00:00, ?it/s]

In [12]:
# Downloads pre-generated spectrogram features (made for microWakeWord in
# particular) for various negative datasets. This can be slow!

output_dir = './negative_datasets'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    link_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
    filenames = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']
    for fname in filenames:
        link = link_root + fname

        zip_path = f"negative_datasets/{fname}"
        !wget -O {zip_path} {link}
        !unzip -q {zip_path} -d {output_dir}

In [ ]:
# Save a yaml config that controls the training process
# These hyperparamters can make a huge different in model quality.
# Experiment with sampling and penalty weights and increasing the number of
# training steps.

import yaml
import os

config = {}

config["window_step_ms"] = 10

config["train_dir"] = (
    "trained_models/wakeword"
)


# Each feature_dir should have at least one of the following folders with this structure:
#  training/
#    ragged_mmap_folders_ending_in_mmap
#  testing/
#    ragged_mmap_folders_ending_in_mmap
#  testing_ambient/
#    ragged_mmap_folders_ending_in_mmap
#  validation/
#    ragged_mmap_folders_ending_in_mmap
#  validation_ambient/
#    ragged_mmap_folders_ending_in_mmap
#
#  sampling_weight: Weight for choosing a spectrogram from this set in the batch
#  penalty_weight: Penalizing weight for incorrect predictions from this set
#  truth: Boolean whether this set has positive samples or negative samples
#  truncation_strategy = If spectrograms in the set are longer than necessary for training, how are they truncated
#       - random: choose a random portion of the entire spectrogram - useful for long negative samples
#       - truncate_start: remove the start of the spectrogram
#       - truncate_end: remove the end of the spectrogram
#       - split: Split the longer spectrogram into separate spectrograms offset by 100 ms. Only for ambient sets

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    # Removed negative_datasets/dinner_party_eval: its truncation_strategy
    # "split" processes whole long ambient clips as many overlapping
    # windows rather than small batched samples (see microwakeword's
    # data.py), independent of batch_size — consistent with training
    # dying at the first eval step with an identical ~3.5GB allocation
    # warning regardless of batch_size (128 and 64 both hit the exact
    # same byte count). It already had sampling_weight=0.0 (monitoring
    # only, never contributed to training), so dropping it costs nothing
    # in model quality, only the ambient-false-positive-rate metric.
]

# Number of training steps in each iteration - various other settings are configured as lists that corresponds to different steps
config["training_steps"] = [10000]

# Penalizing weight for incorrect class predictions - lists that correspond to training steps
config["positive_class_weight"] = [1]
config["negative_class_weight"] = [20]

config["learning_rates"] = [
    0.001,
]  # Learning rates for Adam optimizer - list that corresponds to training steps
# Reduced from 128 -> 64: training was dying with no traceback right at
# step 500 (the first eval checkpoint) twice in a row, right after a
# "exceeds 10% of free system memory" warning for a ~3.5GB allocation —
# consistent with the free-tier Colab instance running out of system RAM
# during evaluation, which batches across all 5 feature sets at once.
# Halving batch size directly reduces that peak allocation.
config["batch_size"] = 64

config["time_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["time_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps
config["freq_mask_max_size"] = [
    0
]  # SpecAugment - list that corresponds to training steps
config["freq_mask_count"] = [0]  # SpecAugment - list that corresponds to training steps

config["eval_step_interval"] = (
    500  # Test the validation sets after every this many steps
)
config["clip_duration_ms"] = (
    1500  # Maximum length of wake word that the streaming model will accept
)

# The best model weights are chosen first by minimizing the specified minimization metric below the specified target_minimization
# Once the target has been met, it chooses the maximum of the maximization metric. Set 'minimization_metric' to None to only maximize
# Available metrics:
#   - "loss" - cross entropy error on validation set
#   - "accuracy" - accuracy of validation set
#   - "recall" - recall of validation set
#   - "precision" - precision of validation set
#   - "false_positive_rate" - false positive rate of validation set
#   - "false_negative_rate" - false negative rate of validation set
#   - "ambient_false_positives" - count of false positives from the split validation_ambient set
#   - "ambient_false_positives_per_hour" - estimated number of false positives per hour on the split validation_ambient set
config["target_minimization"] = 0.9
config["minimization_metric"] = None  # Set to None to disable

config["maximization_metric"] = "average_viable_recall"

with open(os.path.join("training_parameters.yaml"), "w") as file:
    documents = yaml.dump(config, file)

In [14]:
from pathlib import Path
for f in sorted(Path("negative_datasets").rglob("*")):
    if f.is_file():
        print(f"{f}: {f.stat().st_size} bytes")


negative_datasets/dinner_party/training/chime6_train_u01_ch1_mmap/data.ninja: 1166160000 bytes
negative_datasets/dinner_party/training/chime6_train_u01_ch1_mmap/dtype.ninja: 6 bytes
negative_datasets/dinner_party/training/chime6_train_u01_ch1_mmap/ends/data.ninja: 233232 bytes
negative_datasets/dinner_party/training/chime6_train_u01_ch1_mmap/ends/dtype.ninja: 5 bytes
negative_datasets/dinner_party/training/chime6_train_u01_ch1_mmap/ends/order.ninja: 1 bytes
negative_datasets/dinner_party/training/chime6_train_u01_ch1_mmap/ends/shape.ninja: 8 bytes
negative_datasets/dinner_party/training/chime6_train_u01_ch1_mmap/ends/type.ninja: 5 bytes
negative_datasets/dinner_party/training/chime6_train_u01_ch1_mmap/flattened_shapes/data.ninja: 233232 bytes
negative_datasets/dinner_party/training/chime6_train_u01_ch1_mmap/flattened_shapes/dtype.ninja: 5 bytes
negative_datasets/dinner_party/training/chime6_train_u01_ch1_mmap/flattened_shapes/order.ninja: 1 bytes
negative_datasets/dinner_party/training

In [15]:
from mmap_ninja.ragged import RaggedMmap
from pathlib import Path

# Every *_mmap directory under both negative_datasets and your own generated data
for base in ["negative_datasets", "generated_augmented_features"]:
    for p in sorted(Path(base).rglob("*_mmap")):
        try:
            m = RaggedMmap(str(p))
            print(f"OK   {p}  (len={len(m)})")
        except Exception as e:
            print(f"FAIL {p}  ->  {e}")


OK   negative_datasets/dinner_party/training/chime6_train_u01_ch1_mmap  (len=29154)
OK   negative_datasets/dinner_party/training/chime6_train_u02_ch1_mmap  (len=29153)
OK   negative_datasets/dinner_party_eval/testing_ambient/dipco_u01_ch1_mmap  (len=10)
OK   negative_datasets/dinner_party_eval/validation_ambient/chime6_dev_eval_mmap  (len=4)
OK   negative_datasets/no_speech/training/fma_medium_mmap  (len=124900)
OK   negative_datasets/no_speech/training/fsd50k_no_speech_mmap  (len=73797)
OK   negative_datasets/no_speech/training/fsd50k_speech_mmap  (len=8460)
OK   negative_datasets/no_speech/training/wham_train_mmap  (len=31284)
OK   negative_datasets/speech/training/voices_lav_clo_training_mmap  (len=116845)
OK   negative_datasets/speech/training/voices_lav_far_training_mmap  (len=116845)
OK   negative_datasets/speech/training/voices_lav_mid_training_mmap  (len=116845)
OK   generated_augmented_features/testing/wakeword_mmap  (len=100)
OK   generated_augmented_features/training/wakewor

In [16]:
!rm -rf negative_datasets/speech negative_datasets/speech.zip
!wget -O negative_datasets/speech.zip https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/speech.zip
!unzip -q negative_datasets/speech.zip -d negative_datasets


--2026-09-29 13:40:06--  https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/speech.zip
Resolving huggingface.co (huggingface.co)... 18.164.174.118, 18.164.174.23, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.118|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65e327bc1445a768ed343b8c/f80d595a96747a78fc4ab83dde64ae7aa345b36e03ce188dbd1c6683aa87bdef?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27speech.zip%3B+filename%3D%22speech.zip%22%3B&X-Xet-Cas-Uid=public&user_id=public&response-content-type=application%2Fzip&xip=9slEdJKZxK8&Expires=1790692806&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjVlMzI3YmMxNDQ1YTc2OGVkMzQzYjhjL2Y4MGQ1OTVhOTY3NDdhNzhmYzRhYjgzZGRlNjRhZTdhYTM0NWIzNmUwM2NlMTg4ZGJkMWM2NjgzYWE4N2JkZWZcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomWC1YZXQtQ2FzLVVpZD1wdWJsaWMmdXNlcl9pZD1wdWJsaWMmcmVzcG9

In [17]:
from mmap_ninja.ragged import RaggedMmap
m = RaggedMmap("negative_datasets/speech/training/voices_lav_mid_training_mmap")
print("OK, len =", len(m))


OK, len = 116845


In [18]:
import os

def report(path):
    if not os.path.exists(path):
        print(f"{path}: does NOT exist")
    elif os.path.isdir(path):
        n = len(os.listdir(path))
        print(f"{path}: exists, folder, {n} items inside")
    else:
        print(f"{path}: exists, file, {os.path.getsize(path)} bytes")

for p in ["mit_rirs", "audioset_16k", "fma_16k", "generated_augmented_features",
          "generated_augmented_features/training/wakeword_mmap",
          "generated_augmented_features/validation/wakeword_mmap",
          "generated_augmented_features/testing/wakeword_mmap"]:
    report(p)


mit_rirs: exists, folder, 270 items inside
audioset_16k: exists, folder, 0 items inside
fma_16k: exists, folder, 210 items inside
generated_augmented_features: exists, folder, 3 items inside
generated_augmented_features/training/wakeword_mmap: exists, folder, 10 items inside
generated_augmented_features/validation/wakeword_mmap: exists, folder, 10 items inside
generated_augmented_features/testing/wakeword_mmap: exists, folder, 10 items inside


In [19]:
from pathlib import Path

for split in ["training", "validation", "testing"]:
    d = Path(f"generated_augmented_features/{split}/wakeword_mmap")
    print(f"--- {split} ---")
    for f in sorted(d.iterdir()):
        print(f"  {f.name}: {f.stat().st_size} bytes")


--- training ---
  data.ninja: 788480000 bytes
  dtype.ninja: 7 bytes
  ends: 4096 bytes
  flattened_shapes: 4096 bytes
  order.ninja: 1 bytes
  shape.ninja: 8 bytes
  shapes: 4096 bytes
  shapes_are_flat.ninja: 4 bytes
  starts: 4096 bytes
  type.ninja: 6 bytes
--- validation ---
  data.ninja: 49280000 bytes
  dtype.ninja: 7 bytes
  ends: 4096 bytes
  flattened_shapes: 4096 bytes
  order.ninja: 1 bytes
  shape.ninja: 8 bytes
  shapes: 4096 bytes
  shapes_are_flat.ninja: 4 bytes
  starts: 4096 bytes
  type.ninja: 6 bytes
--- testing ---
  data.ninja: 5072000 bytes
  dtype.ninja: 7 bytes
  ends: 4096 bytes
  flattened_shapes: 4096 bytes
  order.ninja: 1 bytes
  shape.ninja: 8 bytes
  shapes: 4096 bytes
  shapes_are_flat.ninja: 4 bytes
  starts: 4096 bytes
  type.ninja: 6 bytes


In [24]:
# Trains a model. When finished, it will quantize and convert the model to a
# streaming version suitable for on-device detection.
# It will resume if stopped, but it will start over at the configured training
# steps in the yaml file.
# Change --train 0 to only convert and test the best-weighted model.
# On Google colab, it doesn't print the mini-batch results, so it may appear
# stuck for several minutes! Additionally, it is very slow compared to training
# on a local GPU.

!python -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

INFO:absl:Loading and analyzing data sets.
2026-09-29 13:50:55.830894: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1790689855.832488   31120 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (128, 204, 40)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (128, 204, 1, 

In [2]:
from pathlib import Path
for p in sorted(Path("trained_models").rglob("*")):
    print(p)


In [3]:
# Downloads the tflite model file. To use on the device, you need to write a
# Model JSON file. See https://esphome.io/components/micro_wake_word for the
# documentation and
# https://github.com/esphome/micro-wake-word-models/tree/main/models/v2 for
# examples. Adjust the probability threshold based on the test results obtained
# after training is finished. You may also need to increase the Tensor arena
# model size if the model fails to load.

from google.colab import files

files.download(f"trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite")

FileNotFoundError: Cannot find file: trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite

In [23]:
from pathlib import Path
for p in sorted(Path("trained_models").rglob("*")):
    print(p)


trained_models/wakeword
trained_models/wakeword/last_weights.weights.h5
trained_models/wakeword/logs
trained_models/wakeword/logs/train
trained_models/wakeword/logs/train/events.out.tfevents.1790688564.a96a65bda8fd.25201.0.v2
trained_models/wakeword/logs/train/events.out.tfevents.1790689435.a96a65bda8fd.29312.0.v2
trained_models/wakeword/logs/validation
trained_models/wakeword/logs/validation/events.out.tfevents.1790688564.a96a65bda8fd.25201.1.v2
trained_models/wakeword/logs/validation/events.out.tfevents.1790689435.a96a65bda8fd.29312.1.v2
trained_models/wakeword/model_summary.txt
trained_models/wakeword/training_config.yaml
